In [1]:
import pandas as pd
import json

In [2]:
!ls /fs/ess/PAS1576/qwjian/verl-s-for-codex/verl-agent/webshop_sft_data/

combine_nl_ct_policy_only_TRAIN_26684.parquet
combine_nl_ct_policy_only_VAL_3472.parquet
converted_from_mixup_for_sft_only_TRAIN_13342.parquet
converted_from_mixup_for_sft_only_VAL_1736.parquet
webshop_2_history_nl_but_memory_policy_only_TRAIN_13342.parquet
webshop_2_history_nl_but_memory_policy_only_VAL_1736.parquet
webshop_baseline_policy_train_13154_dummy_thinking.parquet
webshop_baseline_policy_train_13342.parquet
webshop_baseline_policy_val_1392_dummy_thinking.parquet
webshop_baseline_policy_val_1736.parquet
webshop_KEEP_ACTION_mixed_tasks_train_25342.parquet
webshop_KEEP_ACTION_mixed_tasks_val_3336.parquet
webshop_KEEP_ACTION_policy_only_TRAIN_13342.parquet
webshop_KEEP_ACTION_policy_only_VAL_1736.parquet
webshop_KEEP_ACTION_proxy_tasks_only_12000.parquet
webshop_KEEP_ACTION_proxy_tasks_only_1600.parquet
webshop_mixed_tasks_train_24242_new.parquet
webshop_mixed_tasks_val_3156_new.parquet
webshop_sft_auxiliary_train_11727.parquet
webshop_sft_auxiliary_train_4076.parquet
webshop_sf

In [3]:
# /fs/ess/PAS1576/qwjian/verl-s-for-codex/verl-agent/agent_system/environments/env_package/webshop/webshop/recovered_all_trajectories.jsonl

recovered_all_trajectories = []
with open('/fs/ess/PAS1576/qwjian/verl-s-for-codex/verl-agent/agent_system/environments/env_package/webshop/webshop/recovered_all_trajectories.jsonl', 'r') as f:
    for line in f:
        recovered_all_trajectories.append(json.loads(line))
len(recovered_all_trajectories)


1643

In [4]:
# shuffle and split

import random
random.seed(42)
random.shuffle(recovered_all_trajectories)

train_trajectories = recovered_all_trajectories[:int(len(recovered_all_trajectories) * 0.9)]
val_trajectories = recovered_all_trajectories[int(len(recovered_all_trajectories) * 0.9):]
len(train_trajectories), len(val_trajectories)

(1478, 165)

### BASELINE

In [16]:
# --------------------- WebShop --------------------- #
WEBSHOP_TEMPLATE_NO_HIS = """
You are an expert autonomous agent operating in the WebShop e‑commerce environment. 
Your task is to: {task_description}.
Your current observation is: {current_observation}.
Your admissible actions of the current situation are: 
[
{available_actions}
].

Now it's your turn to take one action for the current step.
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""

WEBSHOP_TEMPLATE = """
You are an expert autonomous agent operating in the WebShop e‑commerce environment.
Your task is to: {task_description}.
Prior to this step, you have already taken {step_count} step(s). Below are the most recent {history_length} observations and the corresponding actions you took: {action_history}
You are now at step {current_step} and your current observation is: {current_observation}.
Your admissible actions of the current situation are: 
[
{available_actions}
].

Now it's your turn to take one action for the current step. 
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""

In [17]:
max_history_length = 2

### Memory

In [18]:
from typing import List, Dict

def make_prompt_input_for_webshop_hard_coded_obs(
    task_description,
    step_count,
    memory_history,
    current_step,
    current_observation,
    available_actions
) -> List[Dict]:
    """Build prompt input with memory history for full training."""
    memory_prompt = []
    for mem in memory_history:
        memory_prompt.append({'type': 'memory_text', 'memory_text': {'text': mem['obs']}})
        memory_prompt.append({'type': 'text', 'text': f"""{mem['act']}"""})

    return [
        {'type': 'text', 'text': f"""You are an expert autonomous agent operating in the WebShop e‑commerce environment.
Your task is to: {task_description}.

=== BEGIN OF MEMORY ===
"""},
        *memory_prompt,
        {'type': 'text', 'text': f"""=== END OF MEMORY ===

You are now at step {current_step} and your current observation is: {current_observation}.
Your admissible actions of the current situation are:
[
{available_actions}
].

Now it's your turn to take one action for the current step.
Before choosing your action, you MUST combine your memories and the current observation to form a comprehensive understanding of the current situation.
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""},
    ]

In [19]:

def make_prompt_input_for_webshop_repeat_obs(
    task_description,
    step_count,
    memory_history,
    current_step,
    current_observation,
    available_actions
) -> List[Dict]:
    """Build prompt input with memory history for full training."""
    memory_prompt = []
    for mem in memory_history:
        memory_prompt.append({'type': 'memory_text', 'memory_text': {'text': mem['obs']}})
        memory_prompt.append({'type': 'text', 'text': f"""{mem['act']}"""})

    return [
        {'type': 'text', 'text': f"""You are an expert autonomous agent operating in the WebShop e‑commerce environment.
Your task is to: {task_description}.

=== BEGIN OF MEMORY ===
"""},
        *memory_prompt,
        {'type': 'text', 'text': f"""=== END OF MEMORY ===

You are now at step {current_step} and your current observation is: """},
    {'type': 'memory_text', 'memory_text': {'text': current_observation}, 'is_memory': False},
    {'type': 'text', 'text': f""" {current_observation}.
Your admissible actions of the current situation are:
[
{available_actions}
].

Now it's your turn to take one action for the current step.
Before choosing your action, you MUST combine your memories and the current observation to form a comprehensive understanding of the current situation.
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""},
    ]

In [20]:

def make_memory_prompt_input_for_webshop_latent_obs(
    task_description,
    step_count,
    memory_history,
    current_step,
    current_observation,
    available_actions
) -> List[Dict]:
    """Build prompt input with memory history for full training."""
    memory_prompt = []
    for mem in memory_history:
        memory_prompt.append({'type': 'memory_text', 'memory_text': {'text': mem['obs']}})
        memory_prompt.append({'type': 'text', 'text': f"""{mem['act']}"""})

    return [
        {'type': 'text', 'text': f"""You are an expert autonomous agent operating in the WebShop e‑commerce environment.
Your task is to: {task_description}.

=== BEGIN OF MEMORY ===
"""},
        *memory_prompt,
        {'type': 'text', 'text': f"""=== END OF MEMORY ===

You are now at step {current_step} and your current observation is: """},
    {'type': 'memory_text', 'memory_text': {'text': current_observation}, 'is_memory': False},
    {'type': 'text', 'text': f"""
Your admissible actions of the current situation are:
[
{available_actions}
].

Now it's your turn to take one action for the current step.
Before choosing your action, you MUST combine your memories and the current observation to form a comprehensive understanding of the current situation.
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.
"""},
    ]

### Build Dataset

In [21]:
from typing import Literal
def build_dataset_from_trajectories(trajectories, use_memory=True, max_history_length=None, 
    memory_type: Literal['hard_coded_obs', 'repeat_obs', 'latent_obs']='hard_coded_obs'
):
    """
    Build training dataset from recovered trajectories.
    
    Args:
        trajectories: List of trajectory dicts with 'steps', 'task_description', etc.
        use_memory: If True, use memory format; otherwise use baseline format
        max_history_length: Max number of history steps to include (None = all)
    
    Returns:
        List of training examples, each with 'input' and 'output'
    """
    if memory_type == 'hard_coded_obs':
        make_prompt_input_for_webshop = make_prompt_input_for_webshop_hard_coded_obs
    elif memory_type == 'repeat_obs':
        make_prompt_input_for_webshop = make_prompt_input_for_webshop_repeat_obs
    elif memory_type == 'latent_obs':
        make_prompt_input_for_webshop = make_memory_prompt_input_for_webshop_latent_obs
    dataset = []
    
    for traj in trajectories:
        task_description = traj['task_description']
        steps = traj['steps']
        traj_id = traj['id']
        
        for i, step in enumerate(steps):
            # Skip the last step (no action to predict)
            if i > 30:
                break
            
            if step['action'] is None or step['action'] == 'noop':
                continue
            
            current_observation = step['obs']
            available_actions = step['available_actions']
            action = step['action']
            
            # Format available actions as string
            #
            available_actions_str = ',\n'.join(f'"{a}"' for a in available_actions)
            
            if use_memory:
                # Build memory history from previous steps
                memory_history = []
                start_idx = 0 if max_history_length is None else max(0, i - max_history_length)
                for j in range(start_idx, i):
                    prev_step = steps[j]
                    if prev_step['action'] and prev_step['action'] != 'noop':
                        memory_history.append({
                            'step_num': prev_step['step_num'],
                            'obs': prev_step['obs'],
                            'act': prev_step['action']
                        })
                
                # Build prompt input
                prompt_input = make_prompt_input_for_webshop(
                    task_description=task_description,
                    step_count=i,
                    memory_history=memory_history,
                    current_step=i,
                    current_observation=current_observation,
                    available_actions=available_actions_str
                )
                
                example = {
                    'id': f"{traj_id}_step{i}",
                    'input': prompt_input,
                    'output': f"<action>{action}</action>",
                    'reward': traj['reward']
                }
            else:
                # Baseline format (no history or simple history)
                if i == 0:
                    # First step - no history
                    prompt = WEBSHOP_TEMPLATE_NO_HIS.format(
                        task_description=task_description,
                        current_observation=current_observation,
                        available_actions=available_actions_str
                    )
                else:
                    # Build action history string
                    history_start = 0 if max_history_length is None else max(0, i - max_history_length)
                    action_history_parts = []
                    for j in range(history_start, i):
                        prev_step = steps[j]
                        if prev_step['action'] and prev_step['action'] != 'noop':
                            # f"[Observation {step_num}: '{obs}', Action {step_num}: '{act}']"
                            action_history_parts.append(
                                f"[Observation {prev_step['step_num']}: '{prev_step['obs']}', Action {prev_step['step_num']}: '{prev_step['action']}']"
                            )
                    action_history = '\n'.join(action_history_parts)
                    
                    prompt = WEBSHOP_TEMPLATE.format(
                        task_description=task_description,
                        step_count=i,
                        history_length=len(action_history_parts),
                        action_history=action_history,
                        current_step=i,
                        current_observation=current_observation,
                        available_actions=available_actions_str
                    )
                
                example = {
                    'id': f"{traj_id}_step{i}",
                    'input': prompt,
                    'output': f"<action>{action}</action>",
                    'reward': traj['reward']
                }
            
            dataset.append(example)
    
    return dataset

In [22]:
# Optional: Build baseline dataset (no memory format)
train_dataset_baseline = build_dataset_from_trajectories(
    train_trajectories, 
    use_memory=False, 
    max_history_length=2  # Keep only last 2 steps in history
)

print(f"Total training examples (baseline format): {len(train_dataset_baseline)}")

eval_dataset_baseline = build_dataset_from_trajectories(
    val_trajectories, 
    use_memory=False, 
    max_history_length=2  # Keep only last 2 steps in history
)

print(f"Total evaluation examples (baseline format): {len(eval_dataset_baseline)}")

Total training examples (baseline format): 13618
Total evaluation examples (baseline format): 1519


In [23]:
def convert_to_llama_factory_format(episode):
    messages = [
        {
            "role": "user",
            "content": episode['input']
        },
        {
            "role": "assistant",
            "content": episode['output']
        }
    ]
    return {
        "meta_info": {
            "id": episode['id'],
            "reward": episode['reward']
        },
        "messages": json.dumps(messages)
    }

train_baseline_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in train_dataset_baseline])
eval_baseline_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in eval_dataset_baseline])

In [24]:
tmp_msg = json.loads(train_baseline_llama_factory.iloc[0]['messages'])

print("Prompt: ", tmp_msg[0]['content'])
print("Output: ", tmp_msg[1]['content'])


Prompt:  
You are an expert autonomous agent operating in the WebShop e‑commerce environment. 
Your task is to: i need a taco seasoning blend that is sugar free, and price lower than 40.00 dollars.
Your current observation is: Instruction: [SEP] i need a taco seasoning blend that is sugar free, and price lower than 40.00 dollars [SEP] Search.
Your admissible actions of the current situation are: 
[
"click[search]",
"search[<query>]"
].

Now it's your turn to take one action for the current step.
Once you've finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.

Output:  <action>search[sugar free taco seasoning blen]</action>


In [25]:
train_baseline_llama_factory.to_parquet(f"new_webshop_baseline_policy_train_llama_factory_no_think.parquet")
eval_baseline_llama_factory.to_parquet(f"new_webshop_baseline_policy_val_llama_factory_no_think.parquet")

# make memory

In [26]:
train_memory_dataset_hard_coded_obs = build_dataset_from_trajectories(
    train_trajectories,
    use_memory=True,
    memory_type='hard_coded_obs'
)
len(train_memory_dataset_hard_coded_obs)


13618

In [27]:
train_memory_dataset_repeat_obs = build_dataset_from_trajectories(
    train_trajectories,
    use_memory=True,
    memory_type='repeat_obs'
)
len(train_memory_dataset_repeat_obs)

train_memory_dataset_latent_obs = build_dataset_from_trajectories(
    train_trajectories,
    use_memory=True,
    memory_type='latent_obs'
)
len(train_memory_dataset_latent_obs)


13618

In [28]:
val_memory_dataset_hard_coded_obs = build_dataset_from_trajectories(
    val_trajectories,
    use_memory=True,
    memory_type='hard_coded_obs'
)
val_memory_dataset_repeat_obs = build_dataset_from_trajectories(
    val_trajectories,
    use_memory=True,
    memory_type='repeat_obs'
)
val_memory_dataset_latent_obs = build_dataset_from_trajectories(
    val_trajectories,
    use_memory=True,
    memory_type='latent_obs'
)
len(val_memory_dataset_hard_coded_obs), len(val_memory_dataset_repeat_obs), len(val_memory_dataset_latent_obs)



(1519, 1519, 1519)

In [30]:
train_memory_dataset_hard_coded_obs_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in train_memory_dataset_hard_coded_obs])
train_memory_dataset_repeat_obs_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in train_memory_dataset_repeat_obs])
train_memory_dataset_latent_obs_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in train_memory_dataset_latent_obs])


In [31]:
tmp_msg = json.loads(train_memory_dataset_hard_coded_obs_llama_factory.iloc[0]['messages'])

print("Prompt: ", tmp_msg[0]['content'])
print("Output: ", tmp_msg[1]['content'])


Prompt:  [{'type': 'text', 'text': 'You are an expert autonomous agent operating in the WebShop e‑commerce environment.\nYour task is to: i need a taco seasoning blend that is sugar free, and price lower than 40.00 dollars.\n\n=== BEGIN OF MEMORY ===\n'}, {'type': 'text', 'text': '=== END OF MEMORY ===\n\nYou are now at step 0 and your current observation is: Instruction: [SEP] i need a taco seasoning blend that is sugar free, and price lower than 40.00 dollars [SEP] Search.\nYour admissible actions of the current situation are:\n[\n"click[search]",\n"search[<query>]"\n].\n\nNow it\'s your turn to take one action for the current step.\nBefore choosing your action, you MUST combine your memories and the current observation to form a comprehensive understanding of the current situation.\nOnce you\'ve finished your reasoning, you should choose an admissible action for current step and present it within <action> </action> tags.\n'}]
Output:  <action>search[sugar free taco seasoning blen]</

In [32]:
train_memory_dataset_hard_coded_obs_llama_factory.to_parquet(f"new_webshop_hard_coded_obs_train_llama_factory_no_think.parquet")
train_memory_dataset_repeat_obs_llama_factory.to_parquet(f"new_webshop_repeat_obs_train_llama_factory_no_think.parquet")
train_memory_dataset_latent_obs_llama_factory.to_parquet(f"new_webshop_latent_obs_train_llama_factory_no_think.parquet")

In [33]:
val_memory_dataset_hard_coded_obs_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in val_memory_dataset_hard_coded_obs])
val_memory_dataset_repeat_obs_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in val_memory_dataset_repeat_obs])
val_memory_dataset_latent_obs_llama_factory = pd.DataFrame([convert_to_llama_factory_format(episode) for episode in val_memory_dataset_latent_obs])


In [ ]:
val_memory_dataset_hard_coded_obs_llama_factory.to_parquet(f"new_webshop_hard_coded_obs_val_llama_factory_no_think.parquet")
val_memory_dataset_repeat_obs_llama_factory.to_parquet(f"new_webshop_repeat_obs_val_llama_factory_no_think.parquet")
val_memory_dataset_latent_obs_llama_factory.to_parquet(f"new_webshop_latent_obs_val_llama_factory_no_think.parquet")